# Week 7 — Robot Sensor Anomaly Detection Benchmark

## Objective

This notebook evaluates anomaly detection methods on synthetic robot sensor data.

Five controlled fault types are injected into normal robot operation:

1. Motor stall
2. Proximity sensor drift
3. Battery degradation
4. IMU axis failure
5. RSSI disruption

Four anomaly detection approaches are benchmarked:

1. Frequency-band energy threshold
2. Isolation Forest
3. One-Class SVM
4. Supervised binary classifier

Performance is evaluated across 10 random seeds using precision, recall, F1 score, and AUROC. The two best-performing methods are further compared using a paired Wilcoxon signed-rank test.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from scipy.stats import wilcoxon

RANDOM_SEEDS = list(range(10))

np.random.seed(42)

## 1. Load Week 2 Synthetic Robot Sensor Data

The synthetic robot sensor datasets generated in Week 2 are loaded below.
Both the 5-minute aggregated dataset and the high-frequency sensor dataset
are inspected before constructing the anomaly injection pipeline.

In [6]:
# Data paths

data_5min_path = "../data/robot_sensor_5min.csv"
data_highfreq_path = "../data/robot_sensor_highfreq.csv"

# Load datasets
df_5min = pd.read_csv(data_5min_path)
df_highfreq = pd.read_csv(data_highfreq_path)

print("5-minute dataset shape:", df_5min.shape)
print("High-frequency dataset shape:", df_highfreq.shape)

5-minute dataset shape: (120960, 12)
High-frequency dataset shape: (48000, 8)


In [7]:
print("=== 5-minute dataset ===")
display(df_5min.head())

print("\nColumns:")
print(df_5min.columns.tolist())

=== 5-minute dataset ===


,timestamp,robot_id,mode,accel_x,accel_y,accel_z,motor_current,proximity,rssi,battery_soc,task_success,imu_magnitude
0,2026-08-10 00:00:00,R01,PATROL,0.181560,-0.022685,9.765057,1.806318,2.234491,-55.888431,93.100778,0,9.766771
1,2026-08-10 00:05:00,R01,PATROL,0.003813,-0.122254,9.707094,2.288695,4.781291,-61.387078,93.089032,1,9.707865
2,2026-08-10 00:10:00,R01,PATROL,0.114086,0.233848,9.832063,2.383943,5.134641,-56.865756,93.065962,1,9.835505
3,2026-08-10 00:15:00,R01,PATROL,0.412985,0.196385,9.728399,2.340912,2.719345,-61.019880,93.026576,1,9.739142
4,2026-08-10 00:20:00,R01,PATROL,0.264022,0.340855,10.063573,2.323048,2.675605,-55.376955,93.016117,1,10.072805



Columns:
['timestamp', 'robot_id', 'mode', 'accel_x', 'accel_y', 'accel_z', 'motor_current', 'proximity', 'rssi', 'battery_soc', 'task_success', 'imu_magnitude']


In [8]:
print("=== High-frequency dataset ===")
display(df_highfreq.head())

print("\nColumns:")
print(df_highfreq.columns.tolist())

=== High-frequency dataset ===


,time,mode,motor_current,accel_x,accel_y,accel_z,imu_magnitude,window_id
0,0.00,PATROL,2.049671,0.060559,0.010018,9.924071,9.924261,0
1,0.05,PATROL,2.050013,-0.042440,-0.003066,9.825160,9.825252,0
2,0.10,PATROL,2.189131,0.131741,0.071587,9.804799,9.805945,0
3,0.15,PATROL,2.330852,0.200482,0.134262,9.774396,9.777374,0
4,0.20,PATROL,2.200534,0.153513,0.121466,9.774075,9.776035,0



Columns:
['time', 'mode', 'motor_current', 'accel_x', 'accel_y', 'accel_z', 'imu_magnitude', 'window_id']


In [9]:
print("=== 5-minute dataset info ===")
df_5min.info()

print("\n=== Missing values ===")
print(df_5min.isna().sum())

=== 5-minute dataset info ===
<class 'pandas.DataFrame'>
RangeIndex: 120960 entries, 0 to 120959
Data columns (total 12 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   timestamp      120960 non-null  str    
 1   robot_id       120960 non-null  str    
 2   mode           120960 non-null  str    
 3   accel_x        120960 non-null  float64
 4   accel_y        120960 non-null  float64
 5   accel_z        120960 non-null  float64
 6   motor_current  120960 non-null  float64
 7   proximity      120960 non-null  float64
 8   rssi           120960 non-null  float64
 9   battery_soc    120960 non-null  float64
 10  task_success   120960 non-null  int64  
 11  imu_magnitude  120960 non-null  float64
dtypes: float64(8), int64(1), str(3)
memory usage: 11.1 MB

=== Missing values ===
timestamp        0
robot_id         0
mode             0
accel_x          0
accel_y          0
accel_z          0
motor_current    0
proximity        0
rssi 

In [10]:
print("=== High-frequency dataset info ===")
df_highfreq.info()

print("\n=== Missing values ===")
print(df_highfreq.isna().sum())

=== High-frequency dataset info ===
<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   time           48000 non-null  float64
 1   mode           48000 non-null  str    
 2   motor_current  48000 non-null  float64
 3   accel_x        48000 non-null  float64
 4   accel_y        48000 non-null  float64
 5   accel_z        48000 non-null  float64
 6   imu_magnitude  48000 non-null  float64
 7   window_id      48000 non-null  int64  
dtypes: float64(6), int64(1), str(1)
memory usage: 2.9 MB

=== Missing values ===
time             0
mode             0
motor_current    0
accel_x          0
accel_y          0
accel_z          0
imu_magnitude    0
window_id        0
dtype: int64
